# Day 23 Sensor Fusion - chay tren Google Colab

Chay cac cell tu tren xuong duoi. Truoc khi bat dau, dua **code Part E-H da hoan thanh** len fork GitHub va dat du lieu tren Google Drive theo cau truc:

```text
MyDrive/Day23-Fusion/Waymo/<segment>.tfrecord
MyDrive/Day23-Fusion/weights/pretrained_fpn-resnet/fpn_resnet_18_epoch_300.pth
```

Notebook cai moi truong, chay test, chay 199 frame voi `--fusion compare --seed 0`, kiem tra metrics/log va sao luu artifacts ve Drive. Project chay detector tren CPU; khong can bat GPU. Khong dua `.tfrecord`, `.pth` hoac `paths.yaml` len GitHub.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/kentranpr4-crypto/K4-L2L3-DAY23-TranCaoQuocDinh-2A202602939-SensorFusion.git'
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/Day23-Fusion')
SEGMENT = 'training_segment-1005081002024129653_5313_150_5333_150_with_camera_labels.tfrecord'
REPO = Path('/content/day23-sensor-fusion')

assert sys.version_info >= (3, 12), f'Can Python >= 3.12, phien nay la {sys.version}'
from google.colab import drive
drive.mount('/content/drive')

if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
else:
    print('Dung repo da clone; neu can code moi, restart runtime hoac git pull thu cong.')
os.environ['DAY23_STUDENT_ROOT'] = str(REPO / 'student')
os.environ['FUSION_LAB_PLATFORM'] = str(REPO / 'platform')
os.environ['DAY23_HEADLESS'] = '1'
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
print('Repo:', REPO)
subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], check=True)

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '-e', str(REPO / 'platform/third_party/waymo_reader'),
    '-e', str(REPO / 'platform'),
    '-e', str(REPO / 'student'),
    'pytest',
], check=True)
subprocess.run(['fusion-run-lab', '--help'], cwd=REPO, check=True)

In [ ]:
import yaml

waymo_dir = DRIVE_DATA_ROOT / 'Waymo'
weights_dir = DRIVE_DATA_ROOT / 'weights'
tfrecord = waymo_dir / SEGMENT
weights = weights_dir / 'pretrained_fpn-resnet/fpn_resnet_18_epoch_300.pth'
assert tfrecord.is_file(), f'Thieu segment: {tfrecord}'
assert weights.is_file(), f'Thieu weights: {weights}'
config = REPO / 'student/config/paths.yaml'
config.write_text(yaml.safe_dump({
    'waymo_dir': str(waymo_dir),
    'weights_dir': str(weights_dir),
    'segment': SEGMENT,
    'frame_start': 0,
    'frame_end': 198,
}, sort_keys=False))
print('Da tao config:', config)
print('Waymo:', tfrecord.stat().st_size, 'bytes; weights:', weights.stat().st_size, 'bytes')

## Kiem tra code truoc khi chay Waymo

Neu cell nay bao con `NotImplementedError`, hoan thanh Part E-H trong fork, cap nhat repo tren Colab, roi chay lai. `pytest` cua de co the hien `xfailed` khi code van la stub, nen notebook kiem tra TODO rieng.

In [ ]:
import re

modules = ('kalman.py', 'camera_fusion.py', 'association.py', 'track_management.py')
pending = [name for name in modules if re.search(
    r'raise\s+NotImplementedError\(',
    (REPO / 'student/workspace' / name).read_text(),
)]
assert not pending, f'Can hoan thanh Part E-H truoc khi chay: {pending}'
subprocess.run([sys.executable, '-m', 'pytest', 'student/tests', '-q'], cwd=REPO, check=True)

## Chay du 199 frame

`compare` chay hai luot tren cung segment: LiDAR-only va LiDAR + camera. Cell nay co the mat thoi gian vi model chay tren CPU va doc du lieu tu Drive. Giu phien Colab hoat dong den khi lenh ket thuc.

In [ ]:
subprocess.run([
    'fusion-run-lab', '--config', str(config),
    '--fusion', 'compare', '--seed', '0',
], cwd=REPO, check=True)

In [ ]:
import json
from datetime import datetime, timezone
from shutil import copy2

sys.path.insert(0, str(REPO / 'platform'))
from fusion_lab.evaluation import read_records, validate_metrics_records

artifacts = REPO / 'student/artifacts'
metrics = json.loads((artifacts / 'metrics.json').read_text())
assert metrics['fusion_mode'] == 'compare' and metrics['frames'] == [0, 198]
assert metrics['seed'] == 0
validate_metrics_records(metrics, read_records(artifacts / 'grade_run.log'))
for mode in ('lidar', 'fused'):
    result = metrics['tracking'][mode]
    print(mode, {key: result[key] for key in (
        'rmse', 'matches', 'ghost_track_frames', 'missed_gt_frames'
    )})

backup = DRIVE_DATA_ROOT / 'runs' / datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-UTC')
backup.mkdir(parents=True, exist_ok=False)
for name in (
    'metrics.json', 'grade_run.log',
    'metrics_lidar.json', 'metrics_fused.json',
    'grade_run_lidar.log', 'grade_run_fused.log',
):
    copy2(artifacts / name, backup / name)
print('Metrics/log hop le. Da sao luu vao:', backup)

## Hoan tat bai nop

Dien so lieu thuc te vao `student/SUBMISSION.md` va tra loi 6 cau hoi bang loi cua ban. Dua 6 file trong thu muc `runs/...` ve `student/artifacts/` cua repo, commit code + artifacts + bao cao, roi chay `python tools/check_submission.py`. Cong cu nay yeu cau artifacts **da duoc git commit**; notebook khong tu push GitHub hoac tao so lieu bao cao. Xem `SUBMISSION.md` o goc repo de nop link va commit hash len LMS.